In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# HM3Dv2 RGB-D：完整轨迹的建图验证

这个实验将真实 RGB-D 和已知相机位姿输入 `graphmap`，验证 **反投影 → 世界系变换 → 体素融合 → 语义关联 → 二维占用投影 → 保存加载**。
默认沿场景 `0000` 的完整导出序列均匀选择 120 帧，按时间顺序融合其中 108 帧，另外 12 帧只用于留出观测检查。
帧文件编号可以不连续；选择依据是 `dataset_wrapper` 的有效样本索引，不自行拼接文件名。

这不是 SLAM：位姿来自数据集，未估计里程计或做回环优化。当前 `Environment` 在每个体素保留最新点，属于累积点云地图，不是 TSDF 表面重建，也未启用动态物体清除。

使用 AIO 内核 `/gray/LAB/envs/AIO/bin/python`，从第一个单元格开始执行。
数据统一由 `dataset_wrapper.datasets.hm3d.HM3D` 读取；并行预取保持输出顺序，不缓存整个场景的像素。
绘图遵循 [scipykit mtp_initializer 协议](/gray/Storage/MyPyLib/scipykit/mtp_initializer/README.md)：
`from scipykit.task.cv import *`、`mfigure`、独立数据 Axes、`label_axes`、命名 `disp`、`plt.close`。图片外置到 `assets`，不在 notebook 内嵌大图。

In [ ]:
from scipykit.task.cv import *

import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from matplotlib.colors import BoundaryNorm, ListedColormap

from dataset_wrapper.datasets.hm3d import HM3D
from graphmap.adapters import rgbd_observation
from graphmap.environment import Environment
from graphmap.misc.config import default_config
from graphmap.pose import Pose
from graphmap.sensor import CameraIntrinsic

ROOT = Path(os.environ.get("GRAPHMAP_HM3D_ROOT", "/R/Datasets/3DGS/hm3dv2"))
SCENE = os.environ.get("GRAPHMAP_HM3D_SCENE", "0000")
N_FRAMES = int(os.environ.get("GRAPHMAP_HM3D_FRAMES", "120"))
PIXEL_STRIDE = int(os.environ.get("GRAPHMAP_HM3D_STRIDE", "8"))
NUM_WORKERS = int(os.environ.get("GRAPHMAP_HM3D_WORKERS", "2"))  # 0 = 同步读取
VOXEL_SIZE = float(os.environ.get("GRAPHMAP_HM3D_RESOLUTION", "0.05"))  # m
DEPTH_SCALE = 0.001  # 此导出保存 uint16 毫米，乘法转换为米
DEPTH_RANGE = (0.15, 8.0)  # axial 深度，端点不包含
OBSTACLE_Z = (0.15, 1.80)  # 楼层相对世界高度；仅用于二维投影
NOTEBOOK_NAME = "04_hm3dv2_rgbd_mapping"

if N_FRAMES < 10 or PIXEL_STRIDE < 1 or NUM_WORKERS < 0:
    raise ValueError("本实验要求至少 10 帧、正像素步长和非负 worker 数")
if not np.isfinite(VOXEL_SIZE) or VOXEL_SIZE <= 0:
    raise ValueError("体素边长必须为有限正数")

# 从仓库根目录或 examples 目录启动都使用相同输出位置。
workdir = Path.cwd()
examples_dir = workdir / "examples" if (workdir / "examples").is_dir() else workdir
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = examples_dir / "outputs" / "hm3dv2_mapping" / SCENE / run_id
output_dir.mkdir(parents=True, exist_ok=True)
figure_options = dict(assets_root=output_dir / "assets", notebook_name=NOTEBOOK_NAME)

print("Python:", sys.executable)
print("Outputs:", output_dir.resolve())

## 1. 输入约定与可重复的帧划分

本机 FNAV 导出已核实：640×640，水平/垂直 FOV 都是 64°，RGB、Depth、Semantic 已对齐；深度是**前向轴深度**，不是射线长度。
`CameraIntrinsic.from_sensor` 接收弧度 FOV，内部处理 graphmap 的负焦距约定：相机局部 `X` 前、`Y` 左、`Z` 上；正深度对应 `X>0`，`hwd.d=-X`。

回放的 `floor_relative_rgb_camera` 是相机到世界的主动变换 `T_world_camera`，已经包含安装外参及楼层基准处理。
这里不再换 Habitat/OpenCV 轴，不再叠加相机安装高度。下面逐帧核对元数据和已验证的配置指纹；不同导出应先核实标定，再修改这些参数与检查。
`record()` 只读索引/标注，不读取 RGB-D 像素。帧划分及相机轨迹也会保存到 CSV。

In [ ]:
dataset = HM3D(ROOT, episodes=[SCENE], load_replay=True)
if len(dataset) < 10:
    raise ValueError("此场景不足 10 帧，无法进行本 notebook 的融合/留出划分")
expected_config_hash = "7570bfa76cd07c69eee812f2b170ac0ef492a190c617e0eb897d797be7617bec"
records = [dataset.record(i) for i in range(len(dataset))]
for record in records:
    metadata = record.metadata
    if metadata["coordinate_frames"]["export_pose"] != "floor_relative_rgb_camera":
        raise ValueError(f"未验证的位姿坐标约定：{record.key}")
    if metadata["config_sha256"]["simulator_config"] != expected_config_hash:
        raise ValueError("此导出的配置未验证，请核对内参/深度单位/坐标后修改本单元格")

camera = CameraIntrinsic.from_sensor(
    width=640, height=640, hfov=np.deg2rad(64), vfov=np.deg2rad(64),
)
all_positions = np.array([record.annotations["pose"]["position"] for record in records])
trajectory_length = float(np.linalg.norm(np.diff(all_positions, axis=0), axis=1).sum())
selected = np.linspace(0, len(dataset) - 1, min(N_FRAMES, len(dataset)), dtype=int)
heldout_indices = selected[5::10]  # 每 10 个选中帧留出第 6 个，不参加融合
mapping_indices = selected[~np.isin(selected, heldout_indices)]
mapping_ds = dataset.iloc[mapping_indices]
heldout_ds = dataset.iloc[heldout_indices]
assert set(mapping_ds.index).isdisjoint(set(heldout_ds.index))
assert len(mapping_ds) + len(heldout_ds) == len(selected)

manifest = pd.DataFrame({
    "dataset_index": np.arange(len(dataset)),
    "episode": [record.key[0] for record in records],
    "frame": [record.key[1] for record in records],
    "split": "unused",
    "camera_x_m": all_positions[:, 0],
    "camera_y_m": all_positions[:, 1],
    "camera_z_m": all_positions[:, 2],
})
manifest.loc[mapping_indices, "split"] = "mapping"
manifest.loc[heldout_indices, "split"] = "heldout"
manifest.to_csv(output_dir / "frame_manifest.csv", index=False)
print(f"{len(dataset)} exported frames; {len(mapping_ds)} mapping + {len(heldout_ds)} held out")
print(f"Camera path: {trajectory_length:.2f} m; fx=fy={camera.fx:.3f} px")
disp(manifest[manifest["split"] != "unused"].head(10))

## 2. 查看输入并统一为 graphmap observation

下面的函数只进行数组/位姿适配，不负责文件读取。`semantic` 可缺失；存在时写入 `IndexDB("semantic_id")`。
HM3D 的这些数值是**实例编号**，不能直接当作类别名；颜色仅用于区分实例，采用 `ID mod 20`，不同实例可能同色。

In [ ]:
def as_observation(sample, *, timestamp=None, with_semantics=True):
    frame = sample["data"]
    pose = sample["annotations"]["pose"]
    T_world_camera = Pose(
        position=pose["position"], rotation=pose["quaternion_xyzw"],
        wrd_frame="world", ego_frame="camera",
    )
    attributes = {}
    if with_semantics and frame["semantic"] is not None:
        attributes["semantic_id"] = frame["semantic"]
    return rgbd_observation(
        frame["depth"], intrinsic=camera, pose_camera=T_world_camera,
        rgb=frame["rgb"], attributes=attributes, timestamp=timestamp,
        depth_scale=DEPTH_SCALE, depth_kind="axial", depth_range=DEPTH_RANGE,
        stride=PIXEL_STRIDE,
    )


first_sample = mapping_ds.iloc[0]
first_frame = first_sample["data"]
first_observation = as_observation(first_sample)
instance_cmap = get_cmap("tab20", 20)
fig = mfigure((12, 3.7), dpi=150, alpha=0)
axes = [fig.add_axes([0.035 + i * 0.325, 0.12, 0.285, 0.76]) for i in range(3)]
axes[0].imshow(first_frame["rgb"])
axes[0].set_title(f"RGB: {first_sample['key'][1]}")
depth_m = first_frame["depth"].astype(float) * DEPTH_SCALE
depth_view = np.ma.masked_where((depth_m <= 0) | ~np.isfinite(depth_m), depth_m)
im = axes[1].imshow(depth_view, vmin=0, vmax=DEPTH_RANGE[1], cmap="viridis")
axes[1].set_title("Axial depth (m)")
fig.colorbar(im, ax=axes[1], fraction=0.045, pad=0.025)
if first_frame["semantic"] is None:
    axes[2].text(0.5, 0.5, "Semantic unavailable", ha="center", va="center")
else:
    axes[2].imshow(instance_cmap(first_frame["semantic"].astype(np.int64) % 20))
axes[2].set_title("Instance ID (color = ID mod 20)")
for ax in axes:
    ax.set(xticks=[], yticks=[])
disp(fig, "01_rgb_depth_instances", **figure_options)
plt.close(fig)

## 3. 按时间顺序融合 RGB-D

二维占用层只接收 `0.15 < world z < 1.80 m` 的点，避免把地板和天花板全部标为障碍。
这是本场景的高度区间假设；多楼层/坡地需要调整。二维地图按相机轨迹及最远视线的保守边界分配，随后检查没有越界点。
三维地图始终保留深度范围内的全部高度。

时间戳使用融合序号 × 0.1 秒，仅用于稳定排序，不表示数据集真实采样频率。
`adapter_ms` 包括反投影、变换和语义表建立；`fusion_ms` 包括三维去重、IndexDB 合并和二维更新。
总耗时还包含 loader 启停、等待读盘、断言和记录开销；本实验不是严格性能基准。

In [ ]:
# 最大 axial 深度转换为最大视线长度，覆盖任意相机旋转后的 XY 范围。
ray_length_factor = np.sqrt(1 + np.tan(camera.hfov / 2)**2 + np.tan(camera.vfov / 2)**2)
half_side = np.max(np.abs(all_positions[:, :2])) + DEPTH_RANGE[1] * ray_length_factor + 1
config = default_config(resolution=VOXEL_SIZE, side_length=float(2 * np.ceil(half_side)))
config.agent = dict(collision_height_min=OBSTACLE_Z[0], collision_height_max=OBSTACLE_Z[1])
env = Environment(config=config, enable_2d_map=True)

frame_rows = []
semantic_frames = 0
mapping_start = perf_counter()
with mapping_ds.loader(num_workers=NUM_WORKERS, batch_size=None, prefetch=2) as samples:
    for order, sample in enumerate(samples):
        assert sample["key"] == records[int(mapping_indices[order])].key
        before_adapter = perf_counter()
        observation = as_observation(sample, timestamp=order * 0.1)
        adapter_ms = (perf_counter() - before_adapter) * 1000
        n_valid = len(observation["points_world"])
        if n_valid == 0:
            raise ValueError(f"此帧没有有效深度，请检查输入范围：{sample['key']}")

        # 独立检查世界变换逆映射、像素与颜色/语义的行对应。
        h, w = observation["pixel_indices"].T
        camera_xyz = (~observation["pose_camera"])(observation["points_world"])
        projected = camera.cam_to_hwd(camera_xyz)
        np.testing.assert_allclose(projected[:, :2], observation["pixel_indices"], atol=2e-3, rtol=0)
        np.testing.assert_allclose(-projected[:, 2], sample["data"]["depth"][h, w] * DEPTH_SCALE,
                                   atol=2e-6, rtol=1e-5)
        np.testing.assert_array_equal(observation["colors"], sample["data"]["rgb"][h, w])
        if sample["data"]["semantic"] is not None:
            semantic_frames += 1
            np.testing.assert_array_equal(
                observation["extra_keys"]["semantic_id"].get_batch(range(n_valid)),
                sample["data"]["semantic"][h, w],
            )

        previous_size = env.N
        before_fusion = perf_counter()
        env.observe(**observation)
        fusion_ms = (perf_counter() - before_fusion) * 1000
        frame_rows.append(dict(
            order=order, dataset_index=int(mapping_indices[order]), frame=sample["key"][1],
            valid_points=n_valid, map_points=env.N, new_voxels=env.N - previous_size,
            adapter_ms=adapter_ms, fusion_ms=fusion_ms,
        ))
        if (order + 1) % 20 == 0 or order + 1 == len(mapping_ds):
            print(f"{order + 1:3d}/{len(mapping_ds)} frames; {env.N:,} map voxels")
mapping_seconds = perf_counter() - mapping_start
frame_stats = pd.DataFrame(frame_rows)
frame_stats.to_csv(output_dir / "mapping_frames.csv", index=False)

xyz = env.infopoints.xyz
colors = env.infopoints.color[:, :3] / 255.0
point_keys = env.point_keys()
semantic_ids = np.asarray(env.get_index_values("semantic_id", default=-1), dtype=np.int64)
assert env.N > 0 and env.frame_count == len(mapping_ds)
assert np.isfinite(xyz).all() and len(np.unique(point_keys)) == env.N
assert (frame_stats["new_voxels"] >= 0).all()  # 当前累积地图没有空域删除
assert env.map2d.is_in_bounds(env.map2d.world_to_grid(xyz[:, :2])).all()
if semantic_frames == len(mapping_ds):
    assert (semantic_ids >= 0).all() and env.index_db("semantic_id").num_links == env.N

print(f"Fused {int(frame_stats.valid_points.sum()):,} samples into {env.N:,} voxels")
print(f"Elapsed {mapping_seconds:.2f} s; semantic frames {semantic_frames}/{len(mapping_ds)}")
disp(frame_stats.describe().loc[["mean", "50%", "max"],
                              ["valid_points", "new_voxels", "adapter_ms", "fusion_ms"]])

In [ ]:
fig = mfigure((11, 3.6), dpi=150, alpha=0)
ax_count = fig.add_axes([0.08, 0.19, 0.39, 0.69])
ax_time = fig.add_axes([0.59, 0.19, 0.38, 0.69])
frame_number = frame_stats["order"] + 1
ax_count.plot(frame_number, frame_stats["valid_points"].cumsum(), label="Input samples")
ax_count.plot(frame_number, frame_stats["map_points"], label="Map voxels")
label_axes(ax_count, "Fused frames", "Count", grid=True, legend=True)
ax_count.set_title("Map growth and voxel reuse")
ax_time.plot(frame_number, frame_stats["adapter_ms"], label="RGB-D adapter")
ax_time.plot(frame_number, frame_stats["fusion_ms"], label="Environment.observe")
label_axes(ax_time, "Fused frames", "Time per frame (ms)", grid=True, legend=True)
ax_time.set_title("Compute time (disk wait excluded)")
disp(fig, "02_growth_and_timing", **figure_options)
plt.close(fig)

## 4. 三维地图与语义关联

为了看清室内结构，图中隐藏 `z≥1.8 m` 的点，并确定性抽样最多 60,000 点；这只影响显示，原始地图没有裁剪。
曲线是**相机**轨迹，不当作机器人机体轨迹。表中 `semantic_id=-1` 表示没有属性关联，其他编号沿用数据集原值。

In [ ]:
unique_ids, counts = np.unique(semantic_ids, return_counts=True)
instance_stats = pd.DataFrame({"semantic_id": unique_ids, "map_voxels": counts})
instance_stats = instance_stats.sort_values("map_voxels", ascending=False)
instance_stats.to_csv(output_dir / "instances.csv", index=False)
disp(instance_stats.head(12))

visible_rows = np.flatnonzero(xyz[:, 2] < OBSTACLE_Z[1])
if len(visible_rows) == 0:
    visible_rows = np.arange(env.N)
plot_rows = visible_rows[np.linspace(0, len(visible_rows) - 1, min(60000, len(visible_rows)), dtype=int)]
plot_xyz = xyz[plot_rows]
instance_colors = instance_cmap(np.maximum(semantic_ids[plot_rows], 0) % 20)
instance_colors[semantic_ids[plot_rows] < 0] = (0.55, 0.55, 0.55, 1)

fig = mfigure((12, 5.3), dpi=150, alpha=0)
for left, point_colors, title in [
    (0.01, colors[plot_rows], "RGB point map (height cutaway)"),
    (0.51, instance_colors, "Instance map (ID mod 20; missing = gray)"),
]:
    ax = fig.add_axes([left, 0.12, 0.46, 0.79], projection="3d")
    ax.scatter(*plot_xyz.T, c=point_colors, s=1.4, linewidths=0, depthshade=False)
    ax.plot(*all_positions.T, color="#d52d87", lw=1, label="Camera path")
    ax.set(xlabel="World x (m)", ylabel="World y (m)", zlabel="World z (m)", title=title)
    ax.set_box_aspect(np.maximum(np.ptp(np.vstack([plot_xyz, all_positions]), axis=0), 1))
    ax.locator_params(axis="z", nbins=2)
    ax.view_init(elev=56, azim=-55)
    ax.legend(loc="upper left")
disp(fig, "03_rgb_and_instance_maps", **figure_options)
plt.close(fig)

## 5. 二维占用地图：直接显示 grid[h, w]

`h` 对应世界 `x`，`w` 对应世界 `y`。不对栅格做转置：图的横轴是世界 `y`、纵轴是世界 `x`。
叠加相机轨迹时同样按横 `y`、纵 `x` 绘制。`extent` 使用格子边界（中心 ± 半格），避免半格偏移。
灰色是未知，深色是指定高度区间内观测到的障碍；本实验没有地面判定、自由空间射线投射或机器人半径膨胀，不能直接作为导航可行地图。

In [ ]:
grid_map = env.map2d
grid = grid_map.grid[:, :, 0]
assert set(np.unique(grid)).issubset({-1.0, 0.0})
height, width = grid.shape
x_edges = (np.array([-0.5, height - 0.5]) - grid_map.origin[0]) * grid_map.resolution
y_edges = (np.array([-0.5, width - 0.5]) - grid_map.origin[1]) * grid_map.resolution
occupancy_cmap = ListedColormap(["#eeeeee", "#263640"])
occupancy_norm = BoundaryNorm([-1.5, -0.5, 0.5], occupancy_cmap.N)

fig = mfigure((8.5, 6.2), dpi=150, alpha=0)
ax = fig.add_axes([0.10, 0.23, 0.69, 0.68])
im = ax.imshow(grid, origin="lower", extent=[*y_edges, *x_edges],
               interpolation="nearest", cmap=occupancy_cmap, norm=occupancy_norm)
ax.plot(all_positions[:, 1], all_positions[:, 0], color="#d52d87", lw=1.1, label="Camera path")
ax.scatter(all_positions[mapping_indices, 1], all_positions[mapping_indices, 0],
           s=8, c="#195ea7", label="Mapping cameras", zorder=3)
ax.scatter(all_positions[heldout_indices, 1], all_positions[heldout_indices, 0],
           s=32, marker="x", c="#d26b00", label="Held-out cameras", zorder=4)
ax.scatter(all_positions[0, 1], all_positions[0, 0], s=70, marker="*", c="#248d44", label="Start", zorder=5)
bounds = np.vstack([xyz[:, :2], all_positions[:, :2]])
ax.set(xlim=(bounds[:, 1].min() - 0.5, bounds[:, 1].max() + 0.5),
       ylim=(bounds[:, 0].min() - 0.5, bounds[:, 0].max() + 0.5),
       title=f"Observed obstacles: {OBSTACLE_Z[0]} < z < {OBSTACLE_Z[1]} m", aspect="equal")
label_axes(ax, "World y (m) / grid w", "World x (m) / grid h")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=9)
cax = fig.add_axes([0.83, 0.28, 0.025, 0.42])
colorbar = fig.colorbar(im, cax=cax, ticks=[-1, 0])
colorbar.ax.set_yticklabels(["Unknown", "Obstacle"])
disp(fig, "04_occupancy_and_camera_path", **figure_options)
plt.close(fig)
print("Grid (h, w):", grid.shape, "observed obstacle cells:", int(np.count_nonzero(grid == 0)))

## 6. 留出 RGB-D 观测的一致性检查

留出的帧从未提交给 `env.observe`。将其有效深度点按各自回放位姿变换到世界系，查询地图中的最近邻欧氏距离：
中位数、P95，以及落在 1 / 2 / 3 个体素边长以内的比例。聚合统计按**观测点数加权**，重复看到同一表面仍算不同观测。

这个单向点到地图指标混合了地图覆盖率、采样密度、体素替换误差及位姿/深度一致性；未见区域会产生大距离。
它不评价完整场景的重建准确率，不是点到真实表面的误差，也不能证明位姿准确。阈值随体素边长变化，比较不同设置时应同时看米制距离和参数。

In [ ]:
tree = cKDTree(xyz)
heldout_rows, distance_chunks = [], []
fused_frame_count = env.frame_count
with heldout_ds.loader(num_workers=NUM_WORKERS, batch_size=None, prefetch=2) as samples:
    for sample in samples:
        heldout_observation = as_observation(sample, with_semantics=False)
        points = heldout_observation["points_world"]
        if len(points) == 0:
            raise ValueError(f"留出帧无有效点：{sample['key']}")
        distances, _ = tree.query(points, k=1)
        assert np.isfinite(distances).all()
        distance_chunks.append(distances)
        heldout_rows.append(dict(
            frame=sample["key"][1], valid_points=len(points),
            median_m=float(np.median(distances)), p95_m=float(np.quantile(distances, 0.95)),
            within_2_voxels=float(np.mean(distances <= 2 * VOXEL_SIZE)),
        ))
assert env.frame_count == fused_frame_count
heldout_stats = pd.DataFrame(heldout_rows)
heldout_stats.to_csv(output_dir / "heldout_frames.csv", index=False)
all_distances = np.concatenate(distance_chunks)
thresholds = VOXEL_SIZE * np.array([1, 2, 3])
coverage = [float(np.mean(all_distances <= threshold)) for threshold in thresholds]
disp(heldout_stats)
print(f"Held-out points: {len(all_distances):,}; median={np.median(all_distances):.4f} m; "
      f"P95={np.quantile(all_distances, 0.95):.4f} m")
for threshold, fraction in zip(thresholds, coverage):
    print(f"Within {threshold:.3f} m: {fraction:.2%}")

fig = mfigure((10.5, 3.8), dpi=150, alpha=0)
ax_cdf = fig.add_axes([0.08, 0.20, 0.38, 0.66])
ax_frame = fig.add_axes([0.58, 0.20, 0.39, 0.66])
sorted_distances = np.sort(all_distances)
ax_cdf.plot(sorted_distances, np.arange(1, len(sorted_distances) + 1) / len(sorted_distances))
for threshold in thresholds:
    ax_cdf.axvline(threshold, color="0.6", lw=0.8, ls="--")
ax_cdf.set(xlim=(0, max(3 * VOXEL_SIZE, np.quantile(all_distances, 0.99))), ylim=(0, 1),
           title="Held-out point-to-map CDF (x to P99 or 3v)")
label_axes(ax_cdf, "Nearest map point distance (m)", "Fraction of held-out points", grid=True)
ax_frame.plot(np.arange(len(heldout_stats)), heldout_stats["within_2_voxels"], marker="o")
ax_frame.set(ylim=(0, 1), title=f"Per-frame fraction within {2 * VOXEL_SIZE:.2f} m")
label_axes(ax_frame, "Held-out frame order", "Fraction of valid points", grid=True)
disp(fig, "05_heldout_consistency", **figure_options)
plt.close(fig)

## 7. 保存、重新加载与结果清单

保存到本次独立运行目录，包含 Environment 快照、逐帧 CSV、输入划分、实例统计、图片和 JSON 指标。
重新加载后逐位比较 `InfoPoints` 数据，以及体素 key、IndexDB 关联和二维栅格。
`InfoPoints` 的颜色/时间列采用位打包，因此用 `uint32` 视图比较，避免把合法打包位模式误当作浮点 NaN。
快照不保存当前相机 pose 或历史队列；这里验证的是地图内容。IndexDB/元数据快照包含 pickle，只加载自己生成的可信文件。

In [ ]:
snapshot_dir = output_dir / "environment"
env.save(snapshot_dir)
restored = Environment.load(snapshot_dir, config=config)
np.testing.assert_array_equal(env.infopoints.data.view(np.uint32), restored.infopoints.data.view(np.uint32))
np.testing.assert_array_equal(point_keys, restored.point_keys())
np.testing.assert_array_equal(semantic_ids, restored.get_index_values("semantic_id", default=-1))
np.testing.assert_array_equal(grid_map.grid, restored.map2d.grid)
np.testing.assert_array_equal(grid_map.origin, restored.map2d.origin)
assert restored.resolution == env.resolution and restored.map2d.resolution == grid_map.resolution
assert restored.frame_count == env.frame_count and set(restored.db) == set(env.db)

summary = {
    "scene": SCENE, "root": str(ROOT.resolve()), "run_id_utc": run_id,
    "python": platform.python_version(), "python_executable": sys.executable,
    "source_metadata": records[0].metadata,
    "intrinsic": {"width": camera.width, "height": camera.height,
                  "fx": camera.fx, "fy": camera.fy, "ppx": camera.ppw, "ppy": camera.pph},
    "depth_scale": DEPTH_SCALE, "depth_kind": "axial", "depth_range_m": DEPTH_RANGE,
    "pixel_stride": PIXEL_STRIDE, "voxel_size_m": VOXEL_SIZE,
    "obstacle_height_m": OBSTACLE_Z, "loader_workers": NUM_WORKERS,
    "exported_frames": len(dataset), "mapping_frames": len(mapping_ds), "heldout_frames": len(heldout_ds),
    "camera_trajectory_length_m": trajectory_length,
    "input_points": int(frame_stats.valid_points.sum()), "map_voxels": env.N,
    "semantic_frames": semantic_frames,
    "semantic_values": int(np.count_nonzero(unique_ids >= 0)),
    "semantic_point_links": int(np.count_nonzero(semantic_ids >= 0)),
    "occupied_grid_cells": int(np.count_nonzero(grid == 0)),
    "mapping_wall_seconds": mapping_seconds,
    "adapter_ms_median": float(frame_stats.adapter_ms.median()),
    "fusion_ms_median": float(frame_stats.fusion_ms.median()),
    "heldout_points": len(all_distances),
    "heldout_median_distance_m": float(np.median(all_distances)),
    "heldout_p95_distance_m": float(np.quantile(all_distances, 0.95)),
    "distance_thresholds_m": thresholds.tolist(), "fractions_within_threshold": coverage,
    "validation": {"pixel_depth_color_semantic_alignment": True, "unique_voxel_keys": True,
                   "grid_bounds": True, "save_load_roundtrip": True},
}
(output_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")
print("PASS: alignment, voxel keys, grid bounds, semantic links, save/load roundtrip")
print(json.dumps({key: summary[key] for key in [
    "mapping_frames", "heldout_frames", "input_points", "map_voxels", "semantic_values",
    "heldout_median_distance_m", "heldout_p95_distance_m", "fractions_within_threshold",
]}, indent=2))
print("Snapshot and report:", output_dir.resolve())

## 如何继续实验

- 减小 `GRAPHMAP_HM3D_STRIDE`（如 8 → 4）增加每帧采样密度；计算和语义表建立成本随之增加。
- 调整 `GRAPHMAP_HM3D_RESOLUTION`（如 0.05 → 0.10）比较地图大小、留出距离和融合耗时。
- 将 `GRAPHMAP_HM3D_FRAMES=506` 可选择本场景全部导出帧，其中每十帧仍留出一帧；分辨率/采样对比时固定帧划分。
- `GRAPHMAP_HM3D_WORKERS=0` 改用当前进程读取；其他预取/内存缓存用法见 [03_hm3d_semantics.ipynb](03_hm3d_semantics.ipynb)。
- `GRAPHMAP_HM3D_SCENE` 切换场景；默认标定指纹检查仍然生效。新传感器/导出格式应先明确单位、相机坐标和 pose 方向。

修改参数后从头重跑。示例使用的高度阈值和抽样密度不适合直接解释为通用导航性能。
后续若需要评价重建精度，应引入独立场景真值及双向距离/可见性评价；若需要未知位姿建图，还需接入定位前端。